**`ingest_tiles`**

Script examples to import tiling polygons (e.g., for data downloads)

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest tiles using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "tiles-osm-2025")',
)
# parser.add_argument(
#     '--admin_ids',
#     help='Administrative unit IDs to ingest (e.g., "US-RI")',
#     nargs='*',
# )
parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped datasets in heap folder after processing',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Global building recipe #1: OpentilesMap
    '--recipe_id tile-obm-2025 '
    # Location #1: Brunswick, NC (flood/hurricane risk case)
    # '--admin_ids US-NC-BS '
    # Processing flags
    '--reprocess '
    # '--redownload '
    '--verbose '
    '--keep_unzipped '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest tile data

In [ ]:
ingester = Ingester(args.recipe_id, verbose=args.verbose)

In [ ]:
ingester.ingest(
    reprocess=args.reprocess,
    redownload=args.redownload,
    keep_unzipped=args.keep_unzipped,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Show full map

In [ ]:
import contextily as cx
import matplotlib.pyplot as plt
import shapely

from openplaces.api import read_entities

if ingester.admin_ids_to_save:
    tiles = read_entities(ingester.recipe, geom=True)

    tiles_3857 = tiles.to_crs('epsg:3857')

    fig, ax = plt.subplots(figsize=(10, 10))

    if isinstance(tiles.geometry.iloc[0], shapely.geometry.Point):
        tiles_3857.plot(ax=ax, facecolor='red', markersize=2, linewidth=0, alpha=0.7)
    elif isinstance(
        tiles.geometry.iloc[0],
        shapely.geometry.Polygon | shapely.geometry.MultiPolygon,
    ):
        if len(tiles) > 250000:
            print('>250K tiles to plot. Taking sample')
            _tiles_plot = tiles_3857.sample(250000)
        else:
            _tiles_plot = tiles_3857
        print(_tiles_plot.crs)
        _tiles_plot.boundary.plot(ax=ax, color='magenta', linewidth=0.5, alpha=0.5)

    ax.set_title(f'{len(tiles):,d} tiles in `{args.recipe_id}`')
    ax.axis('off')
    cx.add_basemap(
        ax,
        crs=tiles_3857.crs,
        source=cx.providers.Esri.WorldImagery,
        alpha=0.5,
    )

## Show random tile with attributes

In [ ]:
from openplaces.viz import show_geometry_context

if ingester.admin_ids_to_save:
    random_tile_id = tiles.sample().index[0]
    print(random_tile_id)
    show_geometry_context(tiles, random_tile_id)